# Esquecimento do histórico com os quatro métodos finais

Este estudo usa somente AST original, Jacobiana, Shao-Wu com `tanh(u) * tanh(v)` nos produtos de lags, e Heaps. Cada método recebe o mesmo banco determinístico de mecanismos em todas as condições; o ruído é gerado em cada passo. Os três notebooks históricos permanecem intactos.

In [1]:
from dataclasses import asdict
from pathlib import Path
import hashlib
import importlib.metadata
import json
import platform
import subprocess
import sys
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = next(p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
            if (p / "src" / "tsfmxai").exists())
sys.path.insert(0, str(ROOT / "src"))
from tsfmxai.evaluation.initialization import (
    GROUPS, InitializationStudyConfig, generator_configs, sample_study_mechanisms,
    run_initialization_cases, run_history_case, run_forgetting_cases, run_history_pair,
)
from tsfmxai.generators.initialization import HistorySpec

CONFIG = InitializationStudyConfig(mechanisms=32, trajectories=3,
                                   mechanism_seed=2026111001, trajectory_seed=2026111002,
                                   innovation_std=.05, example_mechanisms=0)
assert GROUPS == ("Original AST", "Jacobian", "Shao-Wu", "Heaps")
BANKS = {group: sample_study_mechanisms(group, CONFIG) for group in GROUPS}
assert all(len(bank) == CONFIG.mechanisms for bank in BANKS.values())
assert all(m.certificate.holds for group, bank in BANKS.items() if group != "Original AST"
           for m in bank)
print("Mechanism bank:", {group: len(bank) for group, bank in BANKS.items()})

STUDY = "forgetting_initial_history"
results = {group: run_forgetting_cases(group, BANKS[group], CONFIG) for group in GROUPS}
left = HistorySpec("cosine full", "cosine", amplitude=.5, period=48, fraction=1.)
right = HistorySpec("cosine half", "cosine", amplitude=.5, period=48, fraction=.5)
for group in GROUPS:
    result = run_history_pair(group, BANKS[group], CONFIG, left, right,
                              name="cosine full vs half")
    results[group]["rows"].extend(result["rows"])
    results[group]["skipped"].extend(result["skipped"])
rows = pd.DataFrame(row for result in results.values() for row in result["rows"])
skipped = [row for result in results.values() for row in result["skipped"]]
summary = rows.groupby(["method", "case", "horizon"], sort=False).agg(
    pairs=("pair_exploded", "size"), pair_explosions=("pair_exploded", "sum"),
    persistent_forgetting=("persistent_forgetting", "sum"),
    median_final_state_gap=("state_gap", "median"),
).reset_index()
summary["pair_explosion_rate"] = summary.pair_explosions / summary.pairs
summary["forgetting_rate"] = summary.persistent_forgetting / summary.pairs
assert set(summary.method) == set(GROUPS)
assert summary.pairs.eq(CONFIG.mechanisms * CONFIG.trajectories).all()
display(summary)
display(pd.DataFrame(skipped))


Mechanism bank: {'Original AST': 32, 'Jacobian': 32, 'Shao-Wu': 32, 'Heaps': 32}


,method,case,horizon,pairs,pair_explosions,persistent_forgetting,median_final_state_gap,pair_explosion_rate,forgetting_rate
0,Original AST,opposite constants,96,96,27,9,1.000000e+00,0.281250,0.093750
1,Original AST,opposite constants,192,96,48,18,5.385963e-01,0.500000,0.187500
2,Original AST,opposite constants,384,96,51,21,4.402453e-02,0.531250,0.218750
3,Original AST,opposite uniform,96,96,29,11,1.460064e+00,0.302083,0.114583
4,Original AST,opposite uniform,192,96,47,19,3.823519e-01,0.489583,0.197917
5,Original AST,opposite uniform,384,96,50,22,2.094342e-01,0.520833,0.229167
6,Original AST,zero vs uniform,96,96,28,14,7.121268e-01,0.291667,0.145833
7,Original AST,zero vs uniform,192,96,44,20,2.035412e-01,0.458333,0.208333
8,Original AST,zero vs uniform,384,96,49,23,9.548505e-02,0.510417,0.239583
9,Original AST,cosine full vs half,96,96,29,9,9.914449e-01,0.302083,0.093750


""


## Registro completo do experimento

A célula seguinte registra pergunta, hipótese, configuração, procedência, sementes,
métricas, resumo científico e comando de reprodução. Todos os resultados são
observações em horizontes finitos. Uma trajetória inválida permanece no denominador;
casos incompatíveis com o certificado são registrados como não executados.


In [2]:
SOURCE_PATHS = [
    "src/tsfmxai/evaluation/initialization.py",
    "src/tsfmxai/generators/initialization.py",
    "src/tsfmxai/generators/random_expression.py",
    "src/tsfmxai/generators/contraction.py",
    "src/tsfmxai/generators/shao_wu.py",
    "src/tsfmxai/generators/heaps.py",
    "src/tsfmxai/generators/stochastic.py",
    "src/tsfmxai/evaluation/stability.py",
]
record = {
    "experiment_id": '03_forgetting_initial_history',
    "status": "executed",
    "research_question": 'Como pares de históricos iniciais diferentes convergem sob a mesma inovação?',
    "hypothesis": 'Métodos com contração verificada devem reduzir a diferença observada entre históricos admissíveis.',
    "configuration": {
        "study": STUDY,
        "config": asdict(CONFIG),
        "generator_configs": {group: asdict(cfg) for group, cfg in generator_configs(CONFIG).items()},
        "cases": ["opposite constants", "opposite uniform", "zero vs uniform", "cosine full vs half"],
        "noise_policy": "one explicit innovation stream per mechanism and trajectory; uniform bounded for Jacobian, Gaussian for other methods",
        "denominator_policy": "all attempted trajectories or pairs; unsupported startup laws and certified-domain violations listed in skipped",
    },
    "seed_policy": {
        "mechanism_root_seed": CONFIG.mechanism_seed,
        "trajectory_root_seed": CONFIG.trajectory_seed,
        "mechanism": "derive_sample_seed(mechanism_seed, mechanism_index)",
        "initialization_noise_stationary": "SeedSequence([trajectory_seed, mechanism_index, trajectory_index]).spawn(3)",
        "mechanism_bank_reused_across_cases": True,
    },
    "provenance": {
        "git_head": subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=ROOT, text=True).strip(),
        "dirty_worktree_at_execution": bool(subprocess.check_output(["git", "status", "--porcelain"], cwd=ROOT)),
        "python": sys.version,
        "platform": platform.platform(),
        "packages": {p: importlib.metadata.version(p) for p in ("numpy", "pandas", "nbformat", "nbclient")},
        "source_sha256": {p: hashlib.sha256((ROOT/p).read_bytes()).hexdigest() for p in SOURCE_PATHS},
    },
    "metrics": {"summary": json.loads(summary.to_json(orient="records")),
                "skipped": skipped},
    "scientific_summary": "Persistent forgetting at horizon 384: " + "; ".join(f"{g}: {int(summary[summary.method.eq(g) & summary.horizon.eq(384)].persistent_forgetting.sum())}/384 pairs" for g in GROUPS) + ". Failed pairs remain in the denominator.",
    "limitations": "32 mechanisms per family and three paths per mechanism; finite horizons; different innovation laws reflect each method's construction; no claim for arbitrary coupled systems.",
    "reproduction": "From repository root: python -m jupyter nbconvert --to notebook --execute experiments/time_series_initialization/final_methods/03_forgetting_initial_history.ipynb --output 03_forgetting_initial_history.ipynb --output-dir experiments/time_series_initialization/final_methods --ExecutePreprocessor.timeout=1800",
}
print(json.dumps(record, ensure_ascii=False, indent=2, default=lambda v: float(v) if isinstance(v, np.floating) else int(v) if isinstance(v, np.integer) else list(v)))


{
  "experiment_id": "03_forgetting_initial_history",
  "status": "executed",
  "research_question": "Como pares de históricos iniciais diferentes convergem sob a mesma inovação?",
  "hypothesis": "Métodos com contração verificada devem reduzir a diferença observada entre históricos admissíveis.",
  "configuration": {
    "study": "forgetting_initial_history",
    "config": {
      "mechanisms": 32,
      "trajectories": 3,
      "mechanism_seed": 2026111001,
      "trajectory_seed": 2026111002,
      "horizons": [
        96,
        192,
        384
      ],
      "innovation_std": 0.05,
      "explosion_limit": 1000000.0,
      "near_zero_radius": 0.15,
      "tail_steps": 48,
      "flat_tolerance": 0.15,
      "window_steps": 24,
      "constant_value": 0.5,
      "uniform_radius": 1.0,
      "magnitude_scales": [
        0.0,
        0.1,
        0.5,
        1.0,
        2.0,
        4.0
      ],
      "forgetting_tolerance": 1e-06,
      "forgetting_tail": 24,
      "minimum_la